# Session 11: Exploring and visualising with Python

Explore the returns and the basket size of Lakeside Mart. Draw the charts, find the branch where the count and the rate disagree, and write three findings.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Build basket size
Read `orders.csv`, `order_lines.csv`, `returns.csv` and `branches.csv`. Count the lines on each order into `basket_size` and attach it to `orders`.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
lines = pd.read_csv('order_lines.csv')
returns = pd.read_csv('returns.csv')
branches = pd.read_csv('branches.csv')

basket = lines.groupby('order_id').size().rename('basket_size')
orders = orders.merge(basket, on='order_id')
print(orders.shape)

In [ ]:
assert orders.shape == (30229, 8), 'Merge basket_size into orders. It should then have 30,229 rows and 8 columns.'
print('Task 1 correct')

## Task 2. The shape of basket size and order value
Draw a histogram of `basket_size` with one bar per size, then a second histogram of `order_value_ugx`. Store the median basket size in `median_basket`.

In [ ]:
ax = sns.histplot(data=orders, x='basket_size', binwidth=1)
ax.set_title('Lines per order')
plt.show()
ax = sns.histplot(data=orders, x='order_value_ugx')
ax.set_title('Order value, UGX')
plt.show()
median_basket = orders.basket_size.median()
print(median_basket)

In [ ]:
assert median_basket == 2, 'median_basket is the middle value of the basket_size column (use .median())'
print('Task 2 correct')

## Task 3. Basket size and order value
Draw a scatter of `basket_size` against `order_value_ugx` for 2,000 random orders (`random_state=1`), coloured by `payment_method`. Store the correlation, rounded to 2 places, in `r`.

In [ ]:
sample = orders.sample(2000, random_state=1)
sns.scatterplot(data=sample, x='basket_size', y='order_value_ugx', hue='payment_method')
plt.show()
r = round(orders.basket_size.corr(orders.order_value_ugx), 2)
print(r)

In [ ]:
assert 0.25 <= r <= 0.35, 'Use the whole orders table for the correlation'
print('Task 3 correct')

## Task 4. Returns: count and rate
Attach the branch to `returns` and `lines`. Count returns for each branch into `count`, lines into `sold`, and the returns per 100 lines into `rate`, rounded to 2 places.

In [ ]:
returns = returns.merge(orders[['order_id', 'branch_id']], on='order_id')
lines = lines.merge(orders[['order_id', 'branch_id']], on='order_id')
count = returns.groupby('branch_id').size()
sold = lines.groupby('branch_id').size()
rate = (count / sold * 100).round(2)
print(rate.sort_values(ascending=False).head(3))

In [ ]:
assert count['B01'] == 321 and sold['B01'] == 19169, 'Count the rows for each branch_id'
assert rate['B07'] == 3.18, 'rate is returns divided by lines sold, times 100'
print('Task 4 correct')

## Task 5. The reversal
Store the branch with the most returns in `top_count` and the branch with the highest rate in `top_rate`. Draw the two as side by side bars. Print both branch names.

In [ ]:
top_count = count.idxmax()
top_rate = rate.idxmax()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(count.index, count.values)
axes[0].set_title('Number of returns')
axes[1].bar(rate.index, rate.values)
axes[1].set_title('Returns per 100 lines')
plt.show()
names = branches.set_index('branch_id').branch_name
print(names[top_count], names[top_rate])

In [ ]:
assert top_count == 'B01' and top_rate == 'B07', 'Use idxmax on count for top_count and on rate for top_rate'
assert top_count != top_rate, 'The branch leading by count should differ from the branch leading by rate'
print('Task 5 correct')

## Task 6. Seasonality
Total the order value for each month into `monthly` (in millions of UGX) and store the month with the highest total in `peak`. Draw it as a line.

In [ ]:
monthly = orders.groupby(orders.order_date.dt.to_period('M')).order_value_ugx.sum() / 1e6
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monthly.index.astype(str), monthly.values, marker='o')
ax.set_title('Monthly revenue, sample')
ax.set_ylabel('UGX millions')
plt.xticks(rotation=90)
plt.show()
peak = monthly.idxmax()
print(peak, round(monthly.max(), 1))

In [ ]:
assert str(peak) == '2024-12', 'Group by order_date.dt.to_period("M") and take idxmax'
print('Task 6 correct')

## Task 7. Write three findings
In the code cell below, replace each text with one finding of your own. Each needs a number, a comparison and a limit. Use the returns reversal, the December peak and basket size.

In [ ]:
findings = [
    'Fort Portal has 3.18 returns per 100 lines sold, against 2.1 across all branches. This is one sample of orders.',
    'December revenue was 89.5 million in 2024 and 72.8 million in 2025, against 57.9 million in other months. Only two Decembers.',
    'The correlation of basket size and order value is 0.3, a modest link that cannot show cause.',
]
for f in findings:
    print(f)

In [ ]:
assert len(findings) == 3, 'Write exactly three findings'
assert all(any(ch.isdigit() for ch in f) for f in findings), 'Every finding needs a number'
assert not any('Your turn' in f for f in findings), 'Replace each Your turn text with your own finding'
print('Task 7 correct')

## Task 8. Reflect
Write one sentence below. Which of your three findings do you trust least, and what limit would you add to make it safer?